## Setting up a new sensor as science-jubilee toolhead

This notebook will walk through setting up a new sensor!

## Introduction

Making a sensor tool on Jubilee is easy - grab a sensor, bolt it down onto a tool head, and use it!

We are using a modular approach to integrate sensor(s). Each sensor is wired to its own microcontroller node that is then connected to a laptop/raspberry pi for orchestration within the science-jubilee framework.

First, program the node so that it communicates with science-jubilee via serial. The node is a custom pcb with Seeed Xiao as its brain and has a I2C connector.

![Seeed microcontroller](images/microcontroller.png)

### Installing CircuitPython on ESP32 (OPTIONAL EXERCISE)

As of February 2026, we can use the web [CircuitPython installer](https://circuitpython.org/board/seeed_xiao_esp32s3/) (note: browser should support serial communication) to install CircuitPython (v10.0.3).

### Adding CircuitPython firmware

In the `CIRCUITPY` device there should be a `code.py`, a `boot.py`, a `boot.out` (used for debugging only), a `lib` directory, and a `drivers` directory.

The firmware uses a JSON-RPC protocol and has three main parts:

- [`boot.py`](xiao_firmware/boot.py) — runs once at startup. Defines the module identity (`MODULE_ID`, `MODULE_NAME`) and lists which drivers to load. It initializes a `HardwareRegistry` and loads each driver to register sensors and actuators.
- [`drivers/`](xiao_firmware/drivers/) — modular hardware drivers. Each file (e.g. [`i2c_sensors.py`](xiao_firmware/drivers/i2c_sensors.py), [`onboard_led.py`](xiao_firmware/drivers/onboard_led.py)) has a `register(hw)` function that initializes a hardware component and registers it with the `HardwareRegistry`. To add new hardware, create a new driver file and add its name to the `DRIVERS` list in `boot.py`.
- [`code.py`](xiao_firmware/code.py) — the main loop. Handles JSON-RPC serial communication, dispatches commands (e.g. `blink`, `read_sensor`, `get_property`, `set_property`), and returns structured responses.

`code.py` can be shared among microcontrollers while `boot.py` and `drivers/` should be configured per module.

We want to setup AS7341 spectrometer. It comes with its own CircuitPython library, which needs to be added to the `lib` folder first.

***Alternatively***, copy the entire `lib` folder containing all current libraries available for CircuitPython. Adafruit maintains and updates these libraries and the latest bundle can be found [here](https://circuitpython.org/libraries) (note that there might be compatibility issue).

#### Adding firmware

Copy [everything in this folder](xiao_firmware) into CIRCUITPY.

### Making and using a new tool

When working with a new tool, you will need to add tool definition to Science Jubilee. Check [extending-science-jubilee](../extending-science-jubilee/readme.md) for more detail!

The spectral sensor is already added to the science-jubilee library.

In [ ]:
# import science jybilee
from science_jubilee.Machine import Machine
# import the new tool
from science_jubilee.tools.AS7341 import AS7341

In [ ]:
# Connect to the machine
m = Machine(address='192.168.1.2')  # change this to match your machine's IP

In [ ]:
# define the tool
spec = AS7341(index = 0, name = "AS7341 spectral sensor", config = "AS7341")
# load the tool
m.load_tool(spec)

In [ ]:
# pick up the spectral sensor tool!
m.pickup_tool(spec)

In [ ]:
# connect to the Xiao microcontroller over USB serial
spec.connect() 

# blink the onboard LED to confirm connection
spec.blink(1)

In [ ]:
# yay it blinks! now let's take a measurement
# optional: move the sensor around to be closer to the object you want to measure
m.move_to(z=50)
# moving it below z=0 is possible but dangerous!

# turn on the onboard white LED for illumination, take a reading, then turn it off
spec.led_current = 10   # LED brightness (0-150 mA)
spec.led = True
readings = spec.all_channels
spec.led = False

print(readings)

You can save the data for processing later!

In [ ]:
# Take another measurement and save to file
spec.led_current = 10
spec.led = True
spectral_data = spec.all_channels
spec.led = False

# Define the output JSON file path
output_file = "sensor_data.json"

# Save the data to a JSON file
with open(output_file, "w") as json_file:
    json.dump(spectral_data, json_file, indent=4)

In [ ]:
# Misc.
# Dropoff tool!
m.park_tool()